In [9]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('../data/original_data/ais_train.csv', sep='|')


In [10]:
# First sort the dataframe by vesselId and time
train_df = train_df.sort_values(by=['vesselId', 'time']).reset_index(drop=True)

# Create a mask for the first occurrence of each vesselId
first_occurrences = ~train_df['vesselId'].duplicated()

# Calculate the shifted values for steps 1-5
for steps in range(1, 6):
    train_df[f'latitude_{steps}_steps_ago'] = train_df.groupby('vesselId')['latitude'].shift(steps)
    train_df[f'longitude_{steps}_steps_ago'] = train_df.groupby('vesselId')['longitude'].shift(steps)
    train_df[f'time_position_{steps}_steps_ago'] = train_df.groupby('vesselId')['time'].shift(steps)
    
    # Set the values to NaN for the first occurrence of each vesselId
    train_df.loc[first_occurrences, f'latitude_{steps}_steps_ago'] = np.nan
    train_df.loc[first_occurrences, f'longitude_{steps}_steps_ago'] = np.nan
    train_df.loc[first_occurrences, f'time_position_{steps}_steps_ago'] = np.nan

# Calculate max and min changes in last 5 steps
def calculate_max_min_changes(group):
    # Create arrays of previous values including current value
    lat_values = np.array([
        group['latitude'],
        group['latitude_1_steps_ago'],
        group['latitude_2_steps_ago'],
        group['latitude_3_steps_ago'],
        group['latitude_4_steps_ago'],
        group['latitude_5_steps_ago']
    ])
    
    long_values = np.array([
        group['longitude'],
        group['longitude_1_steps_ago'],
        group['longitude_2_steps_ago'],
        group['longitude_3_steps_ago'],
        group['longitude_4_steps_ago'],
        group['longitude_5_steps_ago']
    ])
    
    # Calculate changes between consecutive values
    lat_changes = np.diff(lat_values)
    long_changes = np.diff(long_values)
    
    # Handle cases with NaN values
    lat_changes = lat_changes[~np.isnan(lat_changes)]
    long_changes = long_changes[~np.isnan(long_changes)]
    
    # Return max and min changes (or NaN if no valid changes)
    return pd.Series({
        'max_lat_change_last_5_steps': np.max(lat_changes) if len(lat_changes) > 0 else np.nan,
        'min_lat_change_last_5_steps': np.min(lat_changes) if len(lat_changes) > 0 else np.nan,
        'max_long_change_last_5_steps': np.max(long_changes) if len(long_changes) > 0 else np.nan,
        'min_long_change_last_5_steps': np.min(long_changes) if len(long_changes) > 0 else np.nan
    })

# Apply the calculation to each row
changes = train_df.apply(calculate_max_min_changes, axis=1)
train_df = pd.concat([train_df, changes], axis=1)

train_df = train_df.reset_index(drop=True)



In [11]:
# Verify
print("\nChecking first occurrences of each vesselId:")
for vesselId in train_df['vesselId'].unique()[:2]:  # Check first 2 vessels
    vessel_data = train_df[train_df['vesselId'] == vesselId].head(6)
    print(f"\nVessel {vesselId}:")
    columns_to_show = [
        'latitude', 'longitude',
        'latitude_1_steps_ago', 'longitude_1_steps_ago',
        'latitude_2_steps_ago', 'longitude_2_steps_ago',
        'latitude_3_steps_ago', 'longitude_3_steps_ago',
        'latitude_4_steps_ago', 'longitude_4_steps_ago',
        'latitude_5_steps_ago', 'longitude_5_steps_ago',
        'max_lat_change_last_5_steps', 'min_lat_change_last_5_steps',
        'max_long_change_last_5_steps', 'min_long_change_last_5_steps'
    ]
    print(vessel_data[columns_to_show])


Checking first occurrences of each vesselId:

Vessel 61e9f38eb937134a3c4bfd8b:
   latitude  longitude  latitude_1_steps_ago  longitude_1_steps_ago  \
0   7.50361   77.58340                   NaN                    NaN   
1   7.57302   77.49505               7.50361               77.58340   
2   7.65043   77.39404               7.57302               77.49505   
3   7.71275   77.31394               7.65043               77.39404   
4   7.77191   77.23585               7.71275               77.31394   
5   7.81285   77.18147               7.77191               77.23585   

   latitude_2_steps_ago  longitude_2_steps_ago  latitude_3_steps_ago  \
0                   NaN                    NaN                   NaN   
1                   NaN                    NaN                   NaN   
2               7.50361               77.58340                   NaN   
3               7.57302               77.49505               7.50361   
4               7.65043               77.39404               7

In [12]:
train_df.to_csv('../data/processed_data/train.csv', index=False)
